# GMM clustering notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.ipynb -> dimensionality_reduction.ipynb -> gmm_clustering.ipynb`

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotnine as p9
from plotnine.composition import plot_spacer
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV

from utils.clustering_utils import (
    cluster_summaries,
    predict_clustering_labels,
    print_cluster_stability,
    print_cluster_statistics,
    print_clustering_info,
)
from utils.dimensionality_reduction_utils import BlockPCA
from utils.misc_utils import reduce_dimen_2d, turn_cats_to_continuous
from utils.plotting_utils import (
    make_plot_2d_data,
    make_plot_contingency_table,
    make_plot_cv,
    make_plot_mixture_weights,
    make_plot_silhouette,
)
from utils.saving_utils import load_data, save_plot

pd.options.mode.copy_on_write = True

In [ ]:
# Read in data
data_discov_pca_path = Path("outputs/dim_red/data_discov_pca.pkl")
data_valid_pca_path = Path("outputs/dim_red/data_valid_pca.pkl")
data_tf_pca_path = Path("outputs/dim_red/data_tf_pca.pkl")

data_discov_imputed_path = Path("outputs/imputed/data_discov_imputed.pkl")
data_valid_imputed_path = Path("outputs/imputed/data_valid_imputed.pkl")
data_tf_imputed_path = Path("outputs/imputed/data_tf_imputed.pkl")

# data_discov_reduced_path = Path("data/preprocessed/data_discov_reduced.pkl")
# data_valid_reduced_path = Path("data/preprocessed/data_valid_reduced.pkl")

data_discov_full_path = Path("data/preprocessed/data_discov_full.pkl")
data_valid_full_path = Path("data/preprocessed/data_valid_full.pkl")

vars_info_path = Path("data/preprocessed/vars_info_full.pkl")

pca_discov_path = Path("outputs/dim_red/pca_discov.pkl")
pca_valid_path = Path("outputs/dim_red/pca_valid.pkl")

data_discov_pca: pd.DataFrame = load_data(data_discov_pca_path)
data_valid_pca: pd.DataFrame = load_data(data_valid_pca_path)
data_tf_pca: pd.DataFrame = load_data(data_tf_pca_path)

data_discov_imputed: pd.DataFrame = load_data(data_discov_imputed_path)
data_valid_imputed: pd.DataFrame = load_data(data_valid_imputed_path)
data_tf_imputed: pd.DataFrame = load_data(data_tf_imputed_path)

# data_discov_reduced: pd.DataFrame = load_data(data_discov_reduced_path)
# data_valid_reduced: pd.DataFrame = load_data(data_valid_reduced_path)

data_discov_full: pd.DataFrame = load_data(data_discov_full_path)
data_valid_full: pd.DataFrame = load_data(data_valid_full_path)

vars_info: pd.DataFrame = load_data(vars_info_path)

pca_discov: BlockPCA = load_data(pca_discov_path)
pca_valid: BlockPCA = load_data(pca_valid_path)

# Set outputs folder
outputs_folder = "outputs/clustering"

In [ ]:
# Set up models
def gmm_neg_bic(estimator: GaussianMixture, data: np.typing.NDArray | pd.DataFrame):
    return -estimator.bic(data)


param_grid = [
    # {"n_components": range(1, 50), "covariance_type": ["spherical"]},
    # {"n_components": range(1, 50), "covariance_type": ["tied"]},
    # {"n_components": range(1, 60), "covariance_type": ["diag"]},
    {"n_components": range(1, 40), "covariance_type": ["full"]},
]
gmm_discov = GaussianMixture(random_state=np.random.RandomState(np.random.default_rng(2352).bit_generator))
grid_search_discov = GridSearchCV(gmm_discov, param_grid=param_grid, scoring=gmm_neg_bic, n_jobs=-1, refit=True)

gmm_valid = GaussianMixture(random_state=np.random.RandomState(np.random.default_rng(54645).bit_generator))

# Fit models
grid_search_discov.fit(data_discov_pca)
print(f"Best parameters after cross-validation: {grid_search_discov.best_params_}")
plot_cv = make_plot_cv(grid_search_discov)  # group="covariance_type"
plot_cv.show()
save_plot(plot_cv, Path(outputs_folder, "plot_cv.png"), dpi=150)

gmm_valid.set_params(**grid_search_discov.best_params_)
gmm_valid.fit(data_valid_pca)

model_valid = gmm_valid
model_discov = grid_search_discov.best_estimator_

In [ ]:
# Apply models to predict labels
labels_discov = predict_clustering_labels(model_discov, data_discov_pca)
labels_valid = predict_clustering_labels(model_valid, data_valid_pca)
labels_tf = predict_clustering_labels(model_discov, data_tf_pca)

### Display model information

In [ ]:
# Print information about clusterings
print("Discovery:    ", end="")
print_clustering_info(labels_discov)
print("Validation:   ", end="")
print_clustering_info(labels_valid)
print("Transferred:  ", end="")
print_clustering_info(labels_tf)

# Plot mixture weights
plot_discov_weights = make_plot_mixture_weights(model_discov)
plot_valid_weights = make_plot_mixture_weights(model_valid)

# save_plot(plot_discov_weights, Path(outputs_folder, "plot_discov_weights.png"), dpi=150)
# save_plot(plot_valid_weights, Path(outputs_folder, "plot_valid_weights.png"), dpi=150)

# Combine the two plots
plot_discov_weights += p9.labs(title="Discovery")
plot_valid_weights += p9.labs(title="Validation")
plot_all_weights = plot_discov_weights / plot_valid_weights

plot_all_weights.show()

save_plot(plot_all_weights, Path(outputs_folder, "plot_all_weights.png"), dpi=150)

### Plot clusterings

In [ ]:
# Plot projected clustering in 2d
projection_method = "TSNE"  # projection_method="UMAP"
tsne_kwargs = dict(perplexity=100, early_exaggeration=20)

# Reduce data dimensionality using the specified projection method (time intensive)
data_discov_2d = reduce_dimen_2d(
    data_discov_imputed, projection_method=projection_method, rng=np.random.default_rng(23523), **tsne_kwargs
)
data_valid_2d = reduce_dimen_2d(
    data_valid_imputed, projection_method=projection_method, rng=np.random.default_rng(4086), **tsne_kwargs
)
data_tf_2d = reduce_dimen_2d(
    data_tf_imputed, projection_method=projection_method, rng=np.random.default_rng(6597), **tsne_kwargs
)

# Plot clusterings
clusters_discov_plot = make_plot_2d_data(data=data_discov_2d, cluster_labels=labels_discov)
clusters_valid_plot = make_plot_2d_data(data=data_valid_2d, cluster_labels=labels_valid)
clusters_tf_plot = make_plot_2d_data(data=data_tf_2d, cluster_labels=labels_valid)

# save_plot(clusters_discov_plot, Path(outputs_folder, f"clusters_discov_plot_{projection_method.lower()}.png"), dpi=150)
# save_plot(clusters_valid_plot, Path(outputs_folder, f"clusters_valid_plot_{projection_method.lower()}.png"), dpi=150)
# save_plot(clusters_tf_plot, Path(outputs_folder, f"clusters_tf_plot_{projection_method.lower()}.png"), dpi=150)

# Combine the three plots
clusters_discov_plot += p9.labs(title="Discovery clustering")
clusters_valid_plot += p9.labs(title="Validation clustering")
clusters_tf_plot += p9.labs(title="Transferred clustering")
clusters_all_plot = clusters_discov_plot + plot_spacer() + clusters_valid_plot + clusters_tf_plot

clusters_all_plot.show()

save_plot(clusters_all_plot, Path(outputs_folder, f"clusters_all_plot_{projection_method.lower()}.png"), dpi=150)

### Interpret clusters

In [ ]:
# Backtransform cluster means from PCA space into variable space
pca_discov.inverse_transform(
    data_discov_pca.assign(**{"cluster": labels_discov}).groupby("cluster", observed=True).aggregate("mean")
)

In [ ]:
# Backtransform cluster means from PCA space into variable space
pca_valid.inverse_transform(
    data_valid_pca.assign(**{"cluster": labels_valid}).groupby("cluster", observed=True).aggregate("mean")
)

In [ ]:
# Backtransform cluster means from PCA space into variable space
pca_discov.inverse_transform(
    data_tf_pca.assign(**{"cluster": labels_tf}).groupby("cluster", observed=True).aggregate("mean")
)

In [ ]:
treat_cat_as_continuous = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"]
data_discov_cont = turn_cats_to_continuous(data_discov_full, treat_cat_as_continuous).select_dtypes(exclude="category")
data_valid_cont = turn_cats_to_continuous(data_valid_full, treat_cat_as_continuous).select_dtypes(exclude="category")

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_discov_summaries = cluster_summaries(data_discov_cont, labels_discov, agg_func=["mean", "std"])  # "min", "max"
cluster_discov_summaries

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_valid_summaries = cluster_summaries(data_valid_cont, labels_valid, agg_func=["mean", "std"])  # "min", "max"
cluster_valid_summaries

In [ ]:
# Interpretation of continuous, binary, and ordinal variables
cluster_tf_summaries = cluster_summaries(data_valid_cont, labels_tf, agg_func=["mean", "std"])  # "min", "max"
cluster_tf_summaries

### Silhouette plots

In [ ]:
# Make silhouette plots
silhouette_plot_discov = make_plot_silhouette(data=data_discov_pca, cluster_labels=labels_discov)
silhouette_plot_valid = make_plot_silhouette(data=data_valid_pca, cluster_labels=labels_valid)
silhouette_plot_tf = make_plot_silhouette(data=data_tf_pca, cluster_labels=labels_tf)

# save_plot(silhouette_plot_discov, save_path=Path(outputs_folder, "silhouette_plot_discov.png"), dpi=150)
# save_plot(silhouette_plot_valid, save_path=Path(outputs_folder, "silhouette_plot_valid.png"), dpi=150)
# save_plot(silhouette_plot_tf, save_path=Path(outputs_folder, "silhouette_plot_tf.png"), dpi=150)

# Combine silhouette plots
silhouette_plot_discov += p9.labs(title="Discovery")
silhouette_plot_valid += p9.labs(title="Validation")
silhouette_plot_tf += p9.labs(title="Transferred")
silhouette_plot_all = silhouette_plot_discov + plot_spacer() + silhouette_plot_valid + silhouette_plot_tf

silhouette_plot_all.show()

save_plot(silhouette_plot_all, save_path=Path(outputs_folder, "silhouette_plot_all.png"), dpi=150)

### Compare clusterings
We are dealing with _inferential clustering_, because we are clustering individuals and not variables.\
Method-based validation: The same clustering method is applied to the validation data, and the results are compared between clusterings.\
Result-based validation: The trained cluster algorithm is applied to the validation data, and the results are compared between datasets.\
Cluster stability: The results from the trained and transferred clusterings from the method- and result-based validation steps are compared.

In [ ]:
# Compare clusterings
print_cluster_statistics(
    cluster_labels_list=[labels_discov, labels_valid, labels_tf],
    datas=[data_discov_pca, data_valid_pca, data_tf_pca],
    labels=["discovery", "validation", "transferred"],
)

In [ ]:
# Look at cluster stability between clustering by applying the same method and transferred clustering
print_cluster_stability(labels_tf, labels_valid)

In [ ]:
# Show entailment frequencies using contingency tables (rows add to 1)
plot_cont_table1 = make_plot_contingency_table(
    labels_tf, labels_valid, name1="transferred clustering", name2="validation clustering"
)
plot_cont_table2 = make_plot_contingency_table(
    labels_valid, labels_tf, name1="validation clustering", name2="transferred clustering"
)

# save_plot(plot_cont_table1, save_path=Path(outputs_folder, "plot_cont_table1.png"), dpi=150)
# save_plot(plot_cont_table2, save_path=Path(outputs_folder, "plot_cont_table2.png"), dpi=150)

# Combine the two contingency table plots
plot_cont_table_all = (plot_cont_table1 + plot_cont_table2) & p9.theme(figure_size=(9, 4))

plot_cont_table_all.show()

save_plot(plot_cont_table_all, save_path=Path(outputs_folder, "plot_cont_table_all.png"), dpi=150)